# XCT Segmentation — 3D Viewer

Loads a downsampled version of `full_stack_masks/` (written by
`xct_unet_training.ipynb`) — and optionally the raw stack alongside it —
into napari's 3D view.

**Why downsampled:** the full stack is roughly 4000 x 2028 x 2028 voxels —
about 16.4 billion voxels, ~16GB just for the label volume at 1 byte/voxel,
before even considering the raw grayscale alongside it. That's impractical
to load or render interactively. This notebook builds a much smaller volume
by taking every Nth slice and every Nth pixel within each slice
(`SLICE_STRIDE` / `DOWNSAMPLE_XY` below) — a coarse-but-fast overview, not a
publication-quality render. Increase the strides for a quicker load, decrease
them (at real memory cost) for more detail.

**Downsampling method matters here:** label values (0/1/2) can't be
averaged or interpolated — a pixel value of 0.7 isn't a valid phase. Simple
strided slicing (`arr[::n, ::n]`, i.e. nearest-neighbour) is used throughout
instead of any resizing function that would blend values.


## 0. Setup

In [ ]:
# pip install napari PyQt5 tifffile numpy tqdm

from pathlib import Path

import numpy as np
import tifffile
from tqdm.auto import tqdm


## 1. Config

In [ ]:
CONFIG = {
    # same output_dir as the other notebooks — reads full_stack_masks/
    "output_dir": Path("./xct_segmentation_output"),

    # same original stack used in the annotation/U-Net notebooks — only
    # needed if INCLUDE_RAW is True below
    "stack_path": Path("/path/to/your/stack.tif"),

    "phase_names": ["voidage", "membrane", "polymer_cartridge"],
    "n_classes": 3,

    # --- downsampling ---
    "slice_stride": 4,      # take every Nth slice along z
    "downsample_xy": 4,     # take every Nth pixel within each slice

    # --- what to load ---
    "include_raw": True,    # also load a matching downsampled raw-intensity volume for context

    # --- physical scale (optional) ---
    # if you know your voxel size (e.g. from the Nikon scan profile), set this
    # in physical units (um, mm, whatever) as (z, y, x) — leave as (1, 1, 1)
    # for a relative/unscaled view. Must reflect the ORIGINAL voxel size;
    # the strides above are applied automatically on top of this.
    "voxel_size": (1.0, 1.0, 1.0),
}

masks_dir = CONFIG["output_dir"] / "full_stack_masks"
assert masks_dir.exists(), f"No full_stack_masks/ found at {masks_dir} — run xct_unet_training.ipynb section 11 first."


## 2. Find available mask slices

Section 11 of the U-Net notebook is resumable, so this may run before every
slice is finished — this cell works with whatever's currently on disk and
tells you how much of the stack that covers.


In [ ]:
mask_files = sorted(masks_dir.glob("slice_*.tif"))
mask_indices = [int(p.stem.split("_")[1]) for p in mask_files]

print(f"Found {len(mask_files)} segmented slice(s) in {masks_dir}")
if mask_indices:
    print(f"Index range: {min(mask_indices)} - {max(mask_indices)}")

selected_indices = mask_indices[::CONFIG["slice_stride"]]
print(f"Using every {CONFIG['slice_stride']}th slice -> {len(selected_indices)} slice(s) in the 3D volume")


## 3. Build the downsampled segmentation volume

In [ ]:
def load_mask_slice(idx):
    arr = tifffile.imread(masks_dir / f"slice_{idx:05d}.tif")
    return arr[::CONFIG["downsample_xy"], ::CONFIG["downsample_xy"]]

seg_slices = [load_mask_slice(idx) for idx in tqdm(selected_indices, desc="Loading segmentation")]
seg_volume = np.stack(seg_slices).astype(np.uint8)
del seg_slices

print(f"Segmentation volume shape (Z, Y, X): {seg_volume.shape}")
print(f"Memory: {seg_volume.nbytes / 1e6:.1f} MB")


## 4. Build the matching raw-intensity volume (optional)

Uses the same slice indices and downsample factor as the segmentation, so
the two volumes stay spatially aligned. Skipped entirely if
`CONFIG["include_raw"]` is False.


In [ ]:
if CONFIG["include_raw"]:
    tif_file = tifffile.TiffFile(CONFIG["stack_path"])

    def load_raw_slice(idx):
        arr = tif_file.pages[idx].asarray()
        return arr[::CONFIG["downsample_xy"], ::CONFIG["downsample_xy"]]

    raw_slices = [load_raw_slice(idx) for idx in tqdm(selected_indices, desc="Loading raw")]
    raw_volume = np.stack(raw_slices)
    del raw_slices
    tif_file.close()

    print(f"Raw volume shape (Z, Y, X): {raw_volume.shape}")
    print(f"Memory: {raw_volume.nbytes / 1e6:.1f} MB")
else:
    raw_volume = None
    print("include_raw is False — skipping raw volume.")


## 5. View in napari (3D)

Opens directly in 3D display mode (`viewer.dims.ndisplay = 3`) rather than
the default 2D slice view. Effective voxel spacing accounts for both your
physical `voxel_size` and the strides applied above, so proportions look
correct even though z and xy were downsampled by different amounts.


In [ ]:
%gui qt
import napari

vz, vy, vx = CONFIG["voxel_size"]
scale = (vz * CONFIG["slice_stride"], vy * CONFIG["downsample_xy"], vx * CONFIG["downsample_xy"])

viewer = napari.Viewer(title="XCT segmentation — 3D")

if raw_volume is not None:
    viewer.add_image(raw_volume, name="raw", colormap="gray", scale=scale, opacity=0.5)

viewer.add_labels(seg_volume, name="segmentation", scale=scale, opacity=0.6)

viewer.dims.ndisplay = 3
print("Opened in 3D view. Use the layer list to toggle raw/segmentation, "
      "and the labels layer's controls to hide individual phases.")


## 6. Isolate a single phase (optional)

Useful for looking at e.g. the membrane network on its own, without
voidage/polymer cluttering the view — adds it as a separate toggleable layer
rather than replacing the full segmentation.


In [ ]:
phase_to_isolate = "membrane"  # must match one of CONFIG["phase_names"]
phase_id = CONFIG["phase_names"].index(phase_to_isolate)

isolated = np.where(seg_volume == phase_id, seg_volume, 0)
viewer.add_labels(isolated, name=f"{phase_to_isolate}_only", scale=scale, opacity=0.8)
print(f"Added isolated '{phase_to_isolate}' layer — toggle visibility on the other layers to view it alone.")


## Notes

- This is a fast overview tool, not a publication-quality renderer. For
  polished isosurface renders or movies (e.g. marching-cubes on a single
  phase), PyVista is a good next step — it can consume `seg_volume` directly
  once you're happy with what you're looking at here.
- If `CONFIG["voxel_size"]` is left at `(1, 1, 1)`, proportions are only
  correct if your original voxel spacing was already isotropic (equal in
  all directions) — check your scan profile if that's not the case.
- Increase `slice_stride`/`downsample_xy` further if napari feels sluggish;
  decrease them for more detail once you've found a view worth refining.
